In [12]:
import pandas as pd
import numpy as np

In [13]:
transcript_expression = pd.read_csv('data/K562_GM12878_transcript_tpm.txt', sep='\t')
ioe_events = pd.read_csv('data/events_SE_strict.ioe', sep='\t')
psi_values = pd.read_csv('data/transcript_SE_f1.psi', sep='\t')
ioe_events

,seqname,gene_id,event_id,alternative_transcripts,total_transcripts
0,chr1,ENSG00000228794.8,ENSG00000228794.8;SE:chr1:829104-847654:847806...,ENST00000449005.5,"ENST00000449005.5,ENST00000445118.6"
1,chr1,ENSG00000187583.10,ENSG00000187583.10;SE:chr1:973010-973186:97332...,"ENST00000379410.7,ENST00000379409.6","ENST00000379409.6,ENST00000379407.7,ENST000003..."
2,chr1,ENSG00000162572.20,ENSG00000162572.20;SE:chr1:1284090-1285571:128...,ENST00000325425.12,"ENST00000379101.8,ENST00000338555.6,ENST000003..."
3,chr1,ENSG00000224051.6,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,ENST00000343938.8,"ENST00000464957.1,ENST00000343938.8"
4,chr1,ENSG00000160072.19,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,"ENST00000308647.7,ENST00000472194.6,ENST000004...","ENST00000472194.6,ENST00000485748.5,ENST000003..."
...,...,...,...,...,...
14972,chrY,ENSG00000188120.14,ENSG00000188120.14;SE:chrY:23135273-23138210:2...,ENST00000466332.1,"ENST00000405239.5,ENST00000466332.1"
14973,chrY,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24774421-24784137:2...,ENST00000315357.9,"ENST00000446723.4,ENST00000315357.9"
14974,chrY,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24784208-24786514:2...,ENST00000382365.6,"ENST00000382365.6,ENST00000315357.9"
14975,chrY,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24788965-24791271:2...,"ENST00000382365.6,ENST00000446723.4","ENST00000382365.6,ENST00000446723.4,ENST000003..."


In [14]:
psi_values = psi_values.rename(columns={'K562_TPM': 'K562_SE_psi', 'GM12878_TPM': 'GM12878_SE_psi'})
psi_values['event_id'] = psi_values.index
psi_values.reset_index(drop=True, inplace=True)

psi_values

,K562_SE_psi,GM12878_SE_psi,event_id
0,0.118321,0.029762,ENSG00000000419.12;SE:chr20:50940933-50941129:...
1,0.785425,0.670407,ENSG00000000457.13;SE:chr1:169854964-169855796...
2,0.938394,0.819506,ENSG00000000460.16;SE:chr1:169798958-169800883...
3,0.510837,0.757184,ENSG00000000460.16;SE:chr1:169806088-169807791...
4,0.997132,NaN,ENSG00000000971.15;SE:chr1:196676065-196677476...
...,...,...,...
14972,0.938144,1.000000,ENSG00000285043.1;SE:chr16:30053499-30054784:3...
14973,0.000000,0.000000,ENSG00000285258.1;SE:chr3:63864999-63873749:63...
14974,0.824596,0.132404,ENSG00000285437.1;SE:chr7:102567083-102568010:...
14975,NaN,NaN,ENSG00000285551.1;SE:chr10:62654930-62655367:6...


In [15]:
transcript_expression

,K562_TPM,GM12878_TPM
ENST00000373020.8,0.22,0.03
ENST00000494424.1,0.08,0.00
ENST00000496771.5,0.05,0.00
ENST00000612152.4,0.00,0.00
ENST00000614008.4,0.00,0.00
...,...,...
ENST00000649331.1,0.09,0.00
ENST00000647612.1,0.03,0.00
ENST00000648949.1,0.00,0.00
ENST00000650266.1,0.00,0.00


## Combine all information into an ML response file that we can read in later

In [16]:
k562_summed_tpm = []
gm12878_summed_tpm = []

for i in range(len(ioe_events)):
    event = ioe_events.iloc[i]
    transcripts = event['total_transcripts'].split(',')
    k_tpm_sum = 0
    gm_tpm_sum = 0
    for transcript in transcripts:
        k_tpm_sum += transcript_expression.loc[transcript]['K562_TPM']
        gm_tpm_sum += transcript_expression.loc[transcript]['GM12878_TPM']
    
    gm_tpm_sum = np.log10(gm_tpm_sum + 0.1)
    k_tpm_sum = np.log10(k_tpm_sum + 0.1)
    gm12878_summed_tpm.append(gm_tpm_sum)
    k562_summed_tpm.append(k_tpm_sum)

assert len(k562_summed_tpm) == len(ioe_events) == len(gm12878_summed_tpm)

ioe_events['K562_summed_tpm'] = k562_summed_tpm
ioe_events['GM12878_summed_tpm'] = gm12878_summed_tpm
ioe_events

,seqname,gene_id,event_id,alternative_transcripts,total_transcripts,K562_summed_tpm,GM12878_summed_tpm
0,chr1,ENSG00000228794.8,ENSG00000228794.8;SE:chr1:829104-847654:847806...,ENST00000449005.5,"ENST00000449005.5,ENST00000445118.6",0.846337,0.558709
1,chr1,ENSG00000187583.10,ENSG00000187583.10;SE:chr1:973010-973186:97332...,"ENST00000379410.7,ENST00000379409.6","ENST00000379409.6,ENST00000379407.7,ENST000003...",-0.552842,-0.568636
2,chr1,ENSG00000162572.20,ENSG00000162572.20;SE:chr1:1284090-1285571:128...,ENST00000325425.12,"ENST00000379101.8,ENST00000338555.6,ENST000003...",-0.522879,-0.130768
3,chr1,ENSG00000224051.6,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,ENST00000343938.8,"ENST00000464957.1,ENST00000343938.8",0.499687,1.133219
4,chr1,ENSG00000160072.19,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,"ENST00000308647.7,ENST00000472194.6,ENST000004...","ENST00000472194.6,ENST00000485748.5,ENST000003...",1.236285,1.539202
...,...,...,...,...,...,...,...
14972,chrY,ENSG00000188120.14,ENSG00000188120.14;SE:chrY:23135273-23138210:2...,ENST00000466332.1,"ENST00000405239.5,ENST00000466332.1",-1.000000,-1.000000
14973,chrY,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24774421-24784137:2...,ENST00000315357.9,"ENST00000446723.4,ENST00000315357.9",-1.000000,-1.000000
14974,chrY,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24784208-24786514:2...,ENST00000382365.6,"ENST00000382365.6,ENST00000315357.9",-1.000000,-1.000000
14975,chrY,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24788965-24791271:2...,"ENST00000382365.6,ENST00000446723.4","ENST00000382365.6,ENST00000446723.4,ENST000003...",-1.000000,-1.000000


In [17]:
response = pd.DataFrame({
    'gene_id': ioe_events['gene_id'],
    'event_id': ioe_events['event_id'],
    'K562_summed_tpm': ioe_events['K562_summed_tpm'],
    'GM12878_summed_tpm': ioe_events['GM12878_summed_tpm'],
})

response = pd.merge(response, psi_values, on='event_id', how='left')
response

,gene_id,event_id,K562_summed_tpm,GM12878_summed_tpm,K562_SE_psi,GM12878_SE_psi
0,ENSG00000228794.8,ENSG00000228794.8;SE:chr1:829104-847654:847806...,0.846337,0.558709,0.778902,0.551136
1,ENSG00000187583.10,ENSG00000187583.10;SE:chr1:973010-973186:97332...,-0.552842,-0.568636,NaN,NaN
2,ENSG00000162572.20,ENSG00000162572.20;SE:chr1:1284090-1285571:128...,-0.522879,-0.130768,NaN,NaN
3,ENSG00000224051.6,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,0.499687,1.133219,0.983660,0.971831
4,ENSG00000160072.19,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,1.236285,1.539202,0.861646,0.863808
...,...,...,...,...,...,...
14972,ENSG00000188120.14,ENSG00000188120.14;SE:chrY:23135273-23138210:2...,-1.000000,-1.000000,NaN,NaN
14973,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24774421-24784137:2...,-1.000000,-1.000000,NaN,NaN
14974,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24784208-24786514:2...,-1.000000,-1.000000,NaN,NaN
14975,ENSG00000187191.14,ENSG00000187191.14;SE:chrY:24788965-24791271:2...,-1.000000,-1.000000,NaN,NaN


In [18]:
# remove rows with NaN values
response = response.dropna()
response = response.reset_index(drop=True)
response

,gene_id,event_id,K562_summed_tpm,GM12878_summed_tpm,K562_SE_psi,GM12878_SE_psi
0,ENSG00000228794.8,ENSG00000228794.8;SE:chr1:829104-847654:847806...,0.846337,0.558709,0.778902,0.551136
1,ENSG00000224051.6,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,0.499687,1.133219,0.983660,0.971831
2,ENSG00000160072.19,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,1.236285,1.539202,0.861646,0.863808
3,ENSG00000160072.19,ENSG00000160072.19;SE:chr1:1489274-1489692:148...,1.172019,1.475816,0.270325,0.133512
4,ENSG00000197530.12,ENSG00000197530.12;SE:chr1:1624901-1624991:162...,0.220108,0.382017,0.916667,0.948052
...,...,...,...,...,...,...
7509,ENSG00000071889.16,ENSG00000071889.16;SE:chrX:154507490-154507811...,1.250908,1.129045,0.843679,0.843563
7510,ENSG00000071889.16,ENSG00000071889.16;SE:chrX:154511871-154512823...,1.091667,1.020361,0.977143,0.927746
7511,ENSG00000071889.16,ENSG00000071889.16;SE:chrX:154511871-154512314...,1.184691,1.080626,0.212500,0.193467
7512,ENSG00000130830.14,ENSG00000130830.14;SE:chrX:154792285-154799745...,2.348324,1.070407,0.012292,0.000000


In [19]:
def remove_gene_versions(response_df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove gene versions from the gene_id column in the DataFrame.
    """
    def clean_event(event):
        event_items = event.split(';')
        gene_id = event_items[0]
        gene_id_clean = gene_id.split('.')[0]
        event_items[0] = gene_id_clean
        return ";".join(event_items)

    response_df['gene_id'] = response_df['gene_id'].str.split('.').str[0]
    response_df['event_id'] = response_df['event_id'].apply(clean_event)
    return response_df

clean_response = remove_gene_versions(response.copy())
clean_response

,gene_id,event_id,K562_summed_tpm,GM12878_summed_tpm,K562_SE_psi,GM12878_SE_psi
0,ENSG00000228794,ENSG00000228794;SE:chr1:829104-847654:847806-8...,0.846337,0.558709,0.778902,0.551136
1,ENSG00000224051,ENSG00000224051;SE:chr1:1325102-1326836:132703...,0.499687,1.133219,0.983660,0.971831
2,ENSG00000160072,ENSG00000160072;SE:chr1:1486668-1487863:148791...,1.236285,1.539202,0.861646,0.863808
3,ENSG00000160072,ENSG00000160072;SE:chr1:1489274-1489692:148981...,1.172019,1.475816,0.270325,0.133512
4,ENSG00000197530,ENSG00000197530;SE:chr1:1624901-1624991:162518...,0.220108,0.382017,0.916667,0.948052
...,...,...,...,...,...,...
7509,ENSG00000071889,ENSG00000071889;SE:chrX:154507490-154507811:15...,1.250908,1.129045,0.843679,0.843563
7510,ENSG00000071889,ENSG00000071889;SE:chrX:154511871-154512823:15...,1.091667,1.020361,0.977143,0.927746
7511,ENSG00000071889,ENSG00000071889;SE:chrX:154511871-154512314:15...,1.184691,1.080626,0.212500,0.193467
7512,ENSG00000130830,ENSG00000130830;SE:chrX:154792285-154799745:15...,2.348324,1.070407,0.012292,0.000000


In [20]:
clean_response.to_csv('data/psi_response.csv', index=False)